In [ ]:
from pathlib import Path
from PIL import Image
from collections import Counter
import matplotlib.pyplot as plt
import os
import random
import shutil
#%pip install matplotlib

In [ ]:
CHICKEN_PATH = Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\raw_data\\kury") 
NOT_CHICKEN_PATH = Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\raw_data\\niekury")
print(f"Czy folder CHICKEN_PATH istnieje?: {CHICKEN_PATH.exists()}")

Czy folder CHICKEN_PATH istnieje?: True


In [ ]:
chicken_images = list(CHICKEN_PATH.glob("*"))
not_chicken_images = list(NOT_CHICKEN_PATH.glob("*"))


print(f"Kury: {len(chicken_images)/2}")
print(f"Nie-kury: {len(not_chicken_images)/2}")

Kury: 3243.0
Nie-kury: 2597.0


In [ ]:
extensions = Counter()

for image in chicken_images + not_chicken_images:
    extensions[image.suffix.lower()] += 1

print(extensions)

Counter({'.txt': 5840, '.jpeg': 2988, '.jpg': 2826, '.png': 26})


In [ ]:
sizes = []
dozwolone_rozszerzenia = ('.jpg', '.jpeg', '.png')

for image_path in chicken_images + not_chicken_images:
    # Sprawdzamy, czy plik ma poprawne rozszerzenie (ignorujemy .txt)
    if image_path.suffix.lower() in dozwolone_rozszerzenia:
        try:
            with Image.open(image_path) as img:
                sizes.append(img.size)
        except Exception as e:
            print(f"Błąd podczas otwierania pliku {image_path}: {e}")


In [ ]:
Counter(sizes).most_common(10)

[((224, 224), 2986),
 ((640, 640), 596),
 ((640, 360), 304),
 ((640, 480), 302),
 ((480, 640), 181),
 ((640, 427), 136),
 ((640, 467), 77),
 ((1280, 720), 22),
 ((1600, 1200), 22),
 ((1024, 768), 19)]

In [ ]:
DATASET_PATH = Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\datasetcnn")

TRAIN = 0.70
VAL = 0.15
TEST = 0.15
ROZSZERZENIA_GRAFICZNE = ('.jpg', '.jpeg', '.png')

klasy = {
    "chicken": chicken_images,
    "not_chicken": not_chicken_images
}

for nazwa_klasy, lista_zdjec in klasy.items():
    zdjecia_kopia = [
            p for p in lista_zdjec 
            if p.suffix.lower() in ROZSZERZENIA_GRAFICZNE
        ]
    random.seed(69)
    random.shuffle(zdjecia_kopia)
    
    total_files = len(zdjecia_kopia)
    if total_files == 0:
        print(f"Brak plików do podziału dla klasy: {nazwa_klasy}")
        continue
        
    koniec_train = int(total_files * TRAIN)
    koniec_val = koniec_train + int(total_files * VAL)
    
    train_files = zdjecia_kopia[:koniec_train]
    val_files = zdjecia_kopia[koniec_train:koniec_val]
    test_files = zdjecia_kopia[koniec_val:]
    
    podzialy = {
        "train": train_files,
        "val": val_files,
        "test": test_files
    }
    
    for nazwa_podzialu, pliki in podzialy.items():
        folder_docelowy = Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\datasetcnn\\" + nazwa_podzialu +"\\"+ nazwa_klasy)
        folder_docelowy.mkdir(parents=True, exist_ok=True)
        

        for plik in pliki:
            shutil.copy2(plik, folder_docelowy / plik.name)
            
    print(f"Klasa '{nazwa_klasy}' podzielona pomyślnie!")
    print(f" -> Train: {len(train_files)} | Val: {len(val_files)} | Test: {len(test_files)}")

print(f"\nWszystkie pliki zostały skopiowane do nowej lokalizacji: {DATASET_PATH.resolve()}")


Klasa 'chicken' podzielona pomyślnie!
 -> Train: 2270 | Val: 486 | Test: 487
Klasa 'not_chicken' podzielona pomyślnie!
 -> Train: 1817 | Val: 389 | Test: 391

Wszystkie pliki zostały skopiowane do nowej lokalizacji: C:\Users\klanz\Desktop\MAGISTERKA\datasetcnn


In [ ]:
DATASET_PATH = Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset")


TRAIN = 0.70
VAL = 0.15
TEST = 0.15


klasy = {
    "chicken": chicken_images,
    "not_chicken": not_chicken_images
}

ROZSZERZENIA_GRAFICZNE = ('.jpg', '.jpeg', '.png')

random.seed(69)

for nazwa_klasy, lista_wszystkich_plikow in klasy.items():

    zdjecia = [
        p for p in lista_wszystkich_plikow 
        if p.suffix.lower() in ROZSZERZENIA_GRAFICZNE
    ]
    random.shuffle(zdjecia)

    total_files = len(zdjecia)

    if total_files == 0:
        print(f"Brak plików dla klasy {nazwa_klasy}")
        continue

    koniec_train = int(total_files * TRAIN)
    koniec_val = koniec_train + int(total_files * VAL)

    train_files = zdjecia[:koniec_train]
    val_files = zdjecia[koniec_train:koniec_val]
    test_files = zdjecia[koniec_val:]

    podzialy = {
        "train": train_files,
        "val": val_files,
        "test": test_files
    }

    for nazwa_podzialu, pliki in podzialy.items():

        images_folder = DATASET_PATH / nazwa_podzialu / "images"
        labels_folder = DATASET_PATH / nazwa_podzialu / "labels"

        images_folder.mkdir(parents=True, exist_ok=True)
        labels_folder.mkdir(parents=True, exist_ok=True)

        for obraz in pliki:

            shutil.copy2(obraz, images_folder / obraz.name)

            label = obraz.with_suffix(".txt")

            if label.exists():
                shutil.copy2(label, labels_folder / label.name)
            else:
                print(f"Brak labela: {obraz.name}")

    print(f"\nKlasa: {nazwa_klasy}")
    print(f"Train: {len(train_files)}")
    print(f"Val:   {len(val_files)}")
    print(f"Test:  {len(test_files)}")

print("\nDataset YOLO został utworzony.")



Klasa: chicken
Train: 2270
Val:   486
Test:  487

Klasa: not_chicken
Train: 1817
Val:   389
Test:  391

Dataset YOLO został utworzony.


In [ ]:
modes = Counter()
dozwolone_rozszerzenia = ('.jpg', '.jpeg', '.png')

for image_path in chicken_images + not_chicken_images:
    if image_path.suffix.lower() in dozwolone_rozszerzenia:
        try:
            with Image.open(image_path) as img:
                modes[img.mode] += 1
        except Exception as e:
            print(f"Błąd podczas otwierania obrazu {image_path}: {e}")
print(modes)

Counter({'RGB': 5819, 'RGBA': 17, 'P': 4})


In [ ]:
from PIL import Image
from pathlib import Path

DATASET_PATH = Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset")


naprawione_rgba = 0
naprawione_p = 0

for plik in DATASET_PATH.rglob("*"):
    if plik.is_file() and plik.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp', '.webp']:
        try:
            with Image.open(plik) as img:
                tryb_poczatkowy = img.mode
                
                if tryb_poczatkowy in ['RGBA', 'P']:
                    rgb_img = img.convert('RGB')
                    
                    img.close() 
                    rgb_img.save(plik)
                    
                    if tryb_poczatkowy == 'RGBA':
                        naprawione_rgba += 1
                    else:
                        naprawione_p += 1
        except Exception as e:
            print(f"Błąd przy przetwarzaniu pliku {plik}: {e}")

print("--- PODSUMOWANIE NAPRAWY ---")
print(f"Pomyślnie przekonwertowano zdjęcia RGBA -> RGB: {naprawione_rgba}")
print(f"Pomyślnie przekonwertowano zdjęcia P -> RGB: {naprawione_p}")
print("Wszystkie zdjęcia w folderze 'dataset' mają już spójny format RGB!")


--- PODSUMOWANIE NAPRAWY ---
Pomyślnie przekonwertowano zdjęcia RGBA -> RGB: 17
Pomyślnie przekonwertowano zdjęcia P -> RGB: 4
Wszystkie zdjęcia w folderze 'dataset' mają już spójny format RGB!
